In [59]:
import mitsuba as mi
import matplotlib.pyplot as plt
import math

In [60]:
mi.set_variant('scalar_spectral')

In [66]:
from typing import Optional

def make_sensor(width : int, height : int, wave_length: Optional[float] = None) -> mi.Sensor:
    base = {
        'type': 'perspective',
        'fov': 70,
        'to_world': mi.ScalarTransform4f.look_at(
            origin=[0, 0, 4],
            target=[0, 0, 0],
            up=[0, 1, 0],
        ),
    }
    if wave_length is None:
        film = {'type': 'hdrfilm', 'width': width, 'height': height}
    else:
        film = {
            'type': 'specfilm',
            'width': width,
            'height': height,
            'channel1': {
                'type': 'irregular',
                'wavelengths': f'{wave_length}, {wave_length + 1}',
                'values': '1.0, 1.0',
            },
        }

    return mi.load_dict({**base, 'film': film})

In [67]:
scene = mi.load_dict(mi.cornell_box())

images = dict()
for wave_length in range(380, 780, 40):
    images[wave_length] = mi.render(scene, sensor=make_sensor(1920, 1080, wave_length), spp=256)
images['rgb'] = mi.render(scene, sensor=make_sensor(), spp=256)

TypeError: make_sensor() missing 2 required positional arguments: 'width' and 'height'

In [ ]:
def show_images(images: dict, gamma: float = 2.2, cols: int = 4, cmap=None):
    items = list(images.items())
    n = len(items)
    cols = min(cols, n)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), squeeze=False)

    for ax, (title, image) in zip(axes.flat, items):
        ax.imshow(image ** (1.0 / gamma), cmap=cmap)
        ax.set_title(title)
        ax.axis("off")

    for ax in axes.flat[n:]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()

In [ ]:
show_images(images, cols=3) 